#### Preparation

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision.models import convnext_small, ConvNeXt_Small_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

In [3]:
WEIGHTS_DIR = "../weights"

In [4]:
weights = ConvNeXt_Small_Weights.DEFAULT

train_transform = weights.transforms()
val_transform = weights.transforms()


In [5]:
train_dataset = datasets.ImageFolder(
    r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val",
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)


['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [6]:
model = convnext_small(weights=weights)


In [7]:
in_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(in_features, num_classes)


#### Training 1 (Freeze-Backbone)

In [8]:
for param in model.features.parameters():
    param.requires_grad = False


In [9]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)


ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3
)


In [11]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [12]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

In [ ]:
EPOCHS_STAGE_FREEZE = 8

print("\n-----------Starting Phase 1 Training-----------\n")


for epoch in range(EPOCHS_STAGE_FREEZE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FREEZE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")



-----------Starting Phase 1 Training-----------



Validating: 100%|██████████| 8/8 [01:29<00:00, 11.15s/it]



Epoch [1/8]
Train Loss: 26.7147 | Train Acc: 0.6865
Val Loss: 4.5149 | Val Acc: 0.8375
Precision: 0.8380 | Recall: 0.8375 | F1: 0.8372




Validating: 100%|██████████| 8/8 [06:50<00:00, 51.36s/it]



Epoch [2/8]
Train Loss: 14.6185 | Train Acc: 0.8604
Val Loss: 3.5853 | Val Acc: 0.8542
Precision: 0.8584 | Recall: 0.8542 | F1: 0.8550




...

Validating: 100%|██████████| 8/8 [05:56<00:00, 44.57s/it]



Epoch [6/8]
Train Loss: 7.9767 | Train Acc: 0.9156
Val Loss: 2.4141 | Val Acc: 0.8958
Precision: 0.8997 | Recall: 0.8958 | F1: 0.8950




Validating: 100%|██████████| 8/8 [06:02<00:00, 45.32s/it]



Epoch [7/8]
Train Loss: 7.2096 | Train Acc: 0.9281
Val Loss: 2.1106 | Val Acc: 0.9125
Precision: 0.9151 | Recall: 0.9125 | F1: 0.9125




Training Epoch 8/8:  33%|███▎      | 10/30 [07:03<13:57, 41.88s/it]

#### Training 2 (Fine-Tuning)

In [ ]:
for param in model.parameters():
    param.requires_grad = True


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=1e-5)

In [ ]:
EPOCHS_STAGE_FINETUNE = 20

best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(EPOCHS_STAGE_FINETUNE):

    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_STAGE_FINETUNE)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_STAGE_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "ConvNeXt-S.pth"))

#### Prediction Sample

In [ ]:
checkpoint = torch.load("../weights/ConvNeXt-S.pth")

class_names = checkpoint["class_names"]

model.load_state_dict(checkpoint["model_state"])
model.eval()

def predict(image_path):

    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


In [ ]:
prediction = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\leaf blight\leaf blight_val_13.jpg")
prediction2 = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\leaf spot\leaf spot_val_13.jpg")
prediction3 = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_13.jpg")
print("Healthy Predicted Class:", prediction)
print("Leaf blight Predicted Class:", prediction1)
print("Leaf spot Predicted Class:", prediction2)
print("Spider Mites Predicted Class:", prediction3)
